# Points: group scoring, statements and stress

`points` scores by group (`mode: "group-points"` in info.json): group 1 ("small") is worth 40 points, group 2 ("large") 60. `sol-overflow` is a deliberate near-miss - correct on small inputs, silently wrong on large ones because it adds two 32-bit `int`s that overflow - and it declares exactly that in its own `group_tags` (`{"1": "OK", "2": "WA"}`). `verify` checks that claim against what the solution actually does.

In [1]:
from olympdev.notebook import Workbench

wb = Workbench.open("../example-contest", jobs=4)
points = wb.problem("points")
points.solutions()

['sol-ac', 'sol-overflow']

## Verify: points per group

Same `verify()` as any other problem - but because `points`'s info.json sets `mode: "group-points"`, each solution's column also carries a per-group score, and (for a solution that declares `group_tags`) whether that group's own verdicts matched the claim.

In [2]:
report = points.verify()
report

isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem


isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem


isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem


isolate init with quota failed, retrying without quota: Cannot set disk quota: quotas have not been enabled for this filesystem


Test,sol-ac main OK match cpu/wall 1.0/1.0 ms 100/100 40/40 small 60/60 large,sol-overflow WA WA match cpu/wall 1.0/32 ms 40/100 40/40 small 0/60 large
tests/g1-01.txt,OK 1.0/1.0 ms 472 KB,OK 1.0/32 ms 768 KB
tests/g2-01.txt,OK 1.0/1.0 ms 472 KB,WA 1.0/32 ms 768 KB


The same numbers, read back as plain data - each column's total score and its per-group breakdown:

In [3]:
[
    (
        column.name,
        column.score_text,
        [(g.name, g.score_text, g.tag_reason) for g in column.groups],
    )
    for column in report.columns
]

[('sol-ac', '100/100', [('small', '40/40', None), ('large', '60/60', None)]),
 ('sol-overflow',
  '40/100',
  [('small', '40/40', None), ('large', '0/60', None)])]

## Statement: the subtask table comes from the same groups

`groups` in info.json is also what `statement()` renders as the `\Constraints`/`\SubtaskWithCost` section, through the bundled `olymp.sty` - the card, info.json and the compiled PDF all read the same numbers instead of three copies that can quietly drift apart.

In [4]:
from pathlib import Path


def save_pdf(view, storage, name):
    if view.pdf_digest is None:
        raise RuntimeError(view.failure_reason or "no PDF was produced")
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    path = output_dir / f"{name}.pdf"
    storage.download_file(view.pdf_digest, path)
    return path


statement = points.statement(lang="en")
statement

StatementResultView(status=<PipelineStatus.COMPLETED: 'completed'>, problem_name='points', lang='en', failure_reason=None, wall_seconds=0.4696086269977968, pdf_digest='8774798c871142200e695e8c4e3f2238ec8640616f4578efa10931c864d6de86', jobs=[JobView(job_id='compile', job_name='StatementCompileJob', job_description='Compile a statement source into a PDF', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='c28efc62f773f50b19c66e910889e66675e06fd1b1f144b61cd91b68d5dd65f4', job_digests=[], job_labels=[])])

In [5]:
pdf_path = save_pdf(statement, wb.storage, "points")
print(f"open {pdf_path.resolve()}")

open /home/teraqqq/olympdev/notebooks/output/points.pdf


## Stress: finding the overflow without reading the source

`sol-overflow`'s bug is invisible on group 1's small values but shows up immediately once numbers get large enough to overflow a 32-bit `int` - exactly what group 2's own generator argument (`--group=2`) produces. `stress` is checked against the main solution, the default "model".

In [6]:
found = points.stress("sol-overflow", gen_args="--group=2", iterations=20)
found

StressResultView(status=<PipelineStatus.COMPLETED: 'completed'>, problem_name='points', failure_reason=None, wall_seconds=0.05879627799731679, found=True, iterations_run=1, seed_argv=['--group=2', '1'], verdict=<Verdict.WA: 'WA'>, input_digest='d1ce4e7945ce952476ccdb991bb42483994411ab814635a3eb3d6a5717d6f5d2', candidate_run_digest='6d8d81319ef9ff65bb1307d35ab5d67356db60b3944325547d29a32a93e05180', reference_run_digest='3609c671f3ad68c0c470421d9c906ea19a3a200d8ed54362c9f6e272637cbee3', check_digest='95d47e3e608e7b97822903667fdf91c6e8df24aa2a2380263eb3436161128c98', jobs=[JobView(job_id='compile-checker', job_name='CheckerCompileJob', job_description='Compile checker', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='c1a559b2861f75ec2799f31beeb15412221e5a3712d590dc47727325ac80609d', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'ncmp')]), JobView(job_id='compile-solution', job_name='SolutionCompileJob', job_description='Compile solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='3add8c2cf6e72fd4c115291457c9b0c479342627b3fbeaff2653b842e3a3f691', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'sol-overflow')]), JobView(job_id='compile-model', job_name='ModelSolutionCompileJob', job_description='Compile model solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='be59aad7330187ba384a1ec63b896780331b0b03f0736853f825ec9da84d8b96', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'sol-ac')]), JobView(job_id='compile-generator', job_name='GeneratorCompileJob', job_description='Compile generator', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='0ef2ab188498cad60ddae7af54b37b7cfa4a74f7391473762c12c05fd7945ba7', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'gen')]), JobView(job_id='gen-0', job_name='GeneratorJob', job_description='Run generator', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='ff88b57873faea5bcbdad6ec951d61f194d913feade96c65cf14da362a3dd65e', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'gen')]), JobView(job_id='run-0', job_name='SolutionRunJob', job_description='Run solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='6d8d81319ef9ff65bb1307d35ab5d67356db60b3944325547d29a32a93e05180', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'sol-overflow')]), JobView(job_id='model-0', job_name='ModelSolutionRunJob', job_description='Run model solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='3609c671f3ad68c0c470421d9c906ea19a3a200d8ed54362c9f6e272637cbee3', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'sol-ac')]), JobView(job_id='check-0', job_name='CheckerJob', job_description='Run checker', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='95d47e3e608e7b97822903667fdf91c6e8df24aa2a2380263eb3436161128c98', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'ncmp')])])

In [7]:
print("seed argv:", found.seed_argv)
print("input:    ", wb.storage.download_bytes(found.input_digest).decode().strip())

seed argv: ['--group=2', '1']
input:     2652369200 -2135272688


Group 1 never triggers it - matching `sol-overflow`'s own claim that it's `OK` there and `WA` only on group 2:

In [8]:
clean = points.stress("sol-overflow", gen_args="--group=1", iterations=20)
clean

StressResultView(status=<PipelineStatus.COMPLETED: 'completed'>, problem_name='points', failure_reason=None, wall_seconds=0.5841100380057469, found=False, iterations_run=20, seed_argv=[], verdict=None, input_digest=None, candidate_run_digest=None, reference_run_digest=None, check_digest=None, jobs=[JobView(job_id='compile-checker', job_name='CheckerCompileJob', job_description='Compile checker', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='c1a559b2861f75ec2799f31beeb15412221e5a3712d590dc47727325ac80609d', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'ncmp')]), JobView(job_id='compile-solution', job_name='SolutionCompileJob', job_description='Compile solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='3add8c2cf6e72fd4c115291457c9b0c479342627b3fbeaff2653b842e3a3f691', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'sol-overflow')]), JobView(job_id='compile-model', job_name='ModelSolutionCompileJob', job_description='Compile model solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='be59aad7330187ba384a1ec63b896780331b0b03f0736853f825ec9da84d8b96', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'sol-ac')]), JobView(job_id='compile-generator', job_name='GeneratorCompileJob', job_description='Compile generator', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='0ef2ab188498cad60ddae7af54b37b7cfa4a74f7391473762c12c05fd7945ba7', job_digests=[], job_labels=[('Problem', 'points'), ('Binary', 'gen')]), JobView(job_id='gen-0', job_name='GeneratorJob', job_description='Run generator', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='8f50606bda38e85fc90f996e2cf0fb79318e94734101d954b56d8c579e2918b5', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'gen')]), JobView(job_id='run-0', job_name='SolutionRunJob', job_description='Run solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='fe99f01ee454dace402057f190b10d6c5691ca7d45d417faab013a187aab940d', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'sol-overflow')]), JobView(job_id='model-0', job_name='ModelSolutionRunJob', job_description='Run model solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='99836a95c82a1f7904dac78d248ad92cfccd084d64b6eec36ca6f576dfe7ff5c', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'sol-ac')]), JobView(job_id='check-0', job_name='CheckerJob', job_description='Run checker', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='46d9ae56b57dd1d4c3b9662f29a9b64fe0cdf34d1ae81ac9b9687e94b9dbb2c6', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-1'), ('Binary', 'ncmp')]), JobView(job_id='gen-1', job_name='GeneratorJob', job_description='Run generator', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='85010580537094327fe31483588d8c710092f222ec98fedd80c9e6ebe33e2930', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-2'), ('Binary', 'gen')]), JobView(job_id='run-1', job_name='SolutionRunJob', job_description='Run solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='01bae9faaebf91dbb41a2a211f598fe7d9d9539138d97140046de10c352f0326', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-2'), ('Binary', 'sol-overflow')]), JobView(job_id='model-1', job_name='ModelSolutionRunJob', job_description='Run model solution', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='594a9a76b62a89cd47b8c3a5f7b2e52e91b1cff15403229d9ad72869b468a3ff', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-2'), ('Binary', 'sol-ac')]), JobView(job_id='check-1', job_name='CheckerJob', job_description='Run checker', job_status=<JobStatus.COMPLETED: 'completed'>, job_result='4f2bf36985909975cab67cc0c8187e5627f32966cf40a744831ae561fced489d', job_digests=[], job_labels=[('Problem', 'points'), ('Test', 'iter-2'), ('Binary', 'ncmp')]), JobView(job_id='gen-2', job